# Time series tutorial

In [40]:
import numpy as np
import pandas as pd
import datetime as dt

timesteps = np.arange(273, 356, 8)

In [23]:
from climada.hazard.timeseries import HazardTimeSeries

In [24]:
ts = HazardTimeSeries(timesteps)

In [32]:
ts.check_time_series()

In [49]:
time = pd.date_range("2023-01-01", periods=3, freq="2M")
time

DatetimeIndex(['2023-01-31', '2023-03-31', '2023-05-31'], dtype='datetime64[ns]', freq='2M')

In [50]:
time_ordinal = [t.toordinal() for t in time]
time_ordinal

[738551, 738610, 738671]

In [207]:
def _sample_independent_nevents_per_time_series_bin(
    n_timeseries, n_timesteps, mean_frequency, weights=None, seed=None
):
    """sample number of events per timestep including seasonaily, for n_timeseries time series

    Parameters
    ----------
    n_timeseries : int
        how many time series to sample
    n_timesteps : int
        how many timesteps per time series
    mean_frequency : float
        mean frequency of occurence per timestep
    weights : iterable[float] or None, optional
        1-D array of weights to adapt frequency of single time steps (to simulate seasonailty). Must
        have length n_timesteps (one weight per timestep). By default None, corresponding to balanced weights.
    seed : int, optional
        random seed, by default None

    Returns
    -------
    np.array
        2D array with the number of events per time series (0th dimension) and per timestep (1st dimension)
    """

    if seed is not None:
        np.random.seed(seed)

    if weights is not None:
        if len(weights) != n_timesteps:
            raise ValueError(
                f"Number of timesteps {n_timesteps} must be equal to the length of weights {len(weights)}."
            )
        normalized_weights = np.array(weights) / sum(weights) * len(weights)
        frequency_per_step = normalized_weights * mean_frequency
    else:
        frequency_per_step = np.full(n_timesteps, mean_frequency)

    return np.random.poisson(
        lam=frequency_per_step, size=(n_timeseries, n_timesteps)
    ).astype("int")

In [249]:
test_seed = 23824
# test_seed = None
n_steps = 3
n_ts = 2
test_weights = np.random.random(n_steps)
# test_weights = np.array([i % 2 for i in range(n_steps)])
# test_weights = None
out_array = _sample_independent_nevents_per_time_series_bin(
    n_ts, n_steps, 0.4, weights=test_weights, seed=test_seed
)

In [250]:
timesteps = timesteps[:n_steps]
timesteps

array([273, 281, 289])

## Demonstration on API hazard file

In [343]:
# load example hazard object
from climada.hazard import Hazard, Centroids
from climada.test import get_test_file

haz_tc_fl = Hazard.from_hdf5(
    get_test_file("HAZ_DEMO_FL_15")
)  # Historic tropical cyclones in Florida from 1990 to 2004
haz_tc_fl.check()  # Use always the check() method to see if the hazard has been loaded correctly
# haz_tc_fl_nonzero = haz_tc_fl.select(
#     event_id=haz_tc_fl.event_id[np.where(haz_tc_fl.intensity.sum(axis=1)>0)[0]]
# )

2025-09-12 13:23:39,618 - climada.hazard.io - INFO - Reading /Users/vgebhart/climada/data/hazard/template/HAZ_DEMO_FL_15/v1/HAZ_DEMO_FL_15.h5


In [430]:
timesteps_months = list(range(0, 2 * (365 - 30), 29))

In [431]:
len(timesteps_months)

24

In [433]:
haz_timeseries = HazardTimeSeries.sample_from_hazard_set(
    haz_tc_fl, 100, timesteps_months, seed=1234
)

In [437]:
haz_timeseries.event_name

['timeseries0_hazard0',
 'timeseries0_hazard1',
 'timeseries0_hazard2',
 'timeseries0_hazard3',
 'timeseries1_hazard0',
 'timeseries1_hazard1',
 'timeseries1_hazard2',
 'timeseries2_hazard0',
 'timeseries2_hazard1',
 'timeseries2_hazard2',
 'timeseries2_hazard3',
 'timeseries2_hazard4',
 'timeseries2_hazard5',
 'timeseries3_hazard0',
 'timeseries3_hazard1',
 'timeseries3_hazard2',
 'timeseries3_hazard3',
 'timeseries3_hazard4',
 'timeseries3_hazard5',
 'timeseries3_hazard6',
 'timeseries4_hazard0',
 'timeseries4_hazard1',
 'timeseries4_hazard2',
 'timeseries4_hazard3',
 'timeseries4_hazard4',
 'timeseries4_hazard5',
 'timeseries5_hazard0',
 'timeseries5_hazard1',
 'timeseries5_hazard2',
 'timeseries5_hazard3',
 'timeseries5_hazard4',
 'timeseries5_hazard5',
 'timeseries5_hazard6',
 'timeseries5_hazard7',
 'timeseries6_hazard0',
 'timeseries6_hazard1',
 'timeseries6_hazard2',
 'timeseries6_hazard3',
 'timeseries6_hazard4',
 'timeseries7_hazard0',
 'timeseries7_hazard1',
 'timeseries8_ha

In [427]:
haz_tc_fl.date

array([726670, 726672, 726679, 726681, 726683, 726690, 726703, 726704,
       726714, 726731, 726743, 726746, 726749, 726756, 727012, 727060,
       727068, 727079, 727082, 727083, 727120, 727129, 727130, 727133,
       727309, 727374, 727403, 727426, 727458, 727462, 727463, 727466,
       727467, 727493, 727714, 727732, 727779, 727789, 727797, 727798,
       727813, 727820, 727824, 727835, 728109, 728129, 728154, 728156,
       728169, 728180, 728192, 728195, 728198, 728200, 728234, 728240,
       728447, 728479, 728486, 728502, 728505, 728510, 728513, 728514,
       728527, 728527, 728527, 728531, 728532, 728545, 728548, 728562,
       728563, 728570, 728573, 728586, 728593, 728827, 728845, 728864,
       728890, 728890, 728894, 728897, 728905, 728926, 728936, 728943,
       728946, 728976, 729175, 729205, 729216, 729218, 729222, 729221,
       729270, 729301, 729311, 729597, 729620, 729622, 729625, 729632,
       729640, 729647, 729649, 729651, 729653, 729655, 729667, 729684,
      

In [ ]:
haz_tc_fl.select(event_id=[haz_tc_fl.v[1]])

In [320]:
haz_tc_fl.intensity[0].toarray().sum()

0.0

In [326]:
haz_tc_fl_nonzero = haz_tc_fl.select(
    event_id=haz_tc_fl.event_id[np.where(haz_tc_fl.intensity.sum(axis=1) > 0)[0]]
)

In [ ]:
event_name = [
    f"timeseries{j}_hazard{event_id}"
    for j in range(2)
    for event_id in range(n_events_per_timeseries[j])
]